# Agente basado en LLM + enriquecimiento semántico de avisos inmobiliarios

**Trabajo Final — Deep Learning**  
Maestría en Management & Analytics (MMA), ITBA  
Alumno: **Joaquín Héctor Vassarotto** — Legajo 106442

---

## El problema

Construir modelos de valuación inmobiliaria (AVM) en CABA choca con dos limitaciones de datos:

1. **Adquirir avisos es frágil y manual.** Los scrapers por reglas fijas se rompen ante cada cambio de layout del portal.
2. **La información de valor está en el texto libre.** Estado, amenities, orientación, antigüedad y señales del vendedor viven en la descripción, no en los campos tabulares.

## La solución: dos capas

| Capa | Qué hace | Se entrena? |
|---|---|---|
| **1. Agente (orquestación)** | Un LLM local con patrón ReAct navega ZonaProp, decide cómo paginar y qué extraer, y se recupera ante errores | No — el LLM se usa pre-entrenado |
| **2. Enriquecimiento (NLP)** | Dos transformers BETO fine-tuneados convierten la descripción en variables estructuradas | **Sí — es el núcleo de la materia** |

> El componente evaluado como Deep Learning es la **capa 2**. La capa 1 construye el dataset.

In [ ]:
import json, sys, os
from pathlib import Path

# Permite correr el notebook desde notebooks/ o desde la raiz del repo
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.utils.config import load_config
from src.utils.io import read_jsonl

cfg = load_config()
print("Repo:", ROOT)
print("LLM del agente :", cfg["agent"]["model"])
print("Modelo base NLP:", cfg["ner"]["base_model"])

---
# Capa 1 — El agente

El LLM actúa como *policy*: razona sobre el estado de la página y decide la próxima acción.
Las herramientas que tiene disponibles son las que le permiten interactuar con el navegador.

In [ ]:
from src.agent.react_agent import SYSTEM_PROMPT

print(SYSTEM_PROMPT)

## Métrica 1 del agente: tasa de éxito de extracción

De los avisos que el agente **detecta** en una página de listado, ¿cuántos termina extrayendo
con una descripción utilizable? Se registra durante el scrape real.

In [ ]:
metricas = sorted(Path("reports").glob("agent_metrics*.json"))
if metricas:
    for m in metricas:
        d = json.loads(m.read_text(encoding="utf-8"))
        print(f"--- {m.name}  (modo: {d.get('mode')}) ---")
        for k, v in d["summary"].items():
            print(f"  {k:28s} {v}")
else:
    print("Todavia no se corrio el scrape real (reports/agent_metrics_*.json no existe).")
    print("Para generarlo:  python -m src.agent.run_scrape --mode deterministic --max-listings 300")

## Métrica 2 del agente: robustez ante cambios de layout

Este es el punto que motiva el proyecto. `parser.py` está escrito en **dos niveles**:
primero intenta los selectores `data-qa` del portal y, si fallan, cae a expresiones regulares
sobre el texto plano.

Para medir cuánto aporta ese segundo nivel, tomamos un HTML que el parser sabe leer y le
aplicamos degradaciones que imitan cambios reales del sitio, midiendo qué campos sobreviven.

In [ ]:
from src.agent.robustness import evaluate_fixtures

rob = evaluate_fixtures(ROOT)
print(f"Fuente: {rob['fuente']}\n")
print(f"{'Variante del HTML':26s} {'Retencion de campos':>20s}")
print("-" * 48)
for variante, tasa in rob["retencion_promedio"].items():
    print(f"{variante:26s} {tasa:>20.1%}")

**Lectura del resultado.** Cuando desaparecen los selectores `data-qa`, los campos numéricos
(precio, ambientes, dormitorios, baños, antigüedad, expensas) y el barrio se siguen recuperando
por regex sobre el texto plano. Lo único que se pierde es la **descripción**, que necesita al
menos un gancho de markup reconocible para poder aislarse del resto de la página.

Un scraper de un solo nivel habría caído a 0% en ese escenario.

---
# Capa 2 — Los modelos entrenados

## Esquema de etiquetado

Dos tareas distintas sobre la misma descripción:

- **NER (token classification)** con esquema BIO: dónde, dentro del texto, se menciona cada atributo.
- **Clasificación multilabel**: qué señales del vendedor tiene el aviso en conjunto.

Son multilabel y no multiclase porque un aviso puede ser a la vez "dueño directo" y "urgencia".

In [ ]:
from src.annotation.label_schema import ENTITY_TYPES, BIO_LABELS, SIGNAL_CLASSES

print("Entidades NER :", ENTITY_TYPES)
print("Etiquetas BIO :", BIO_LABELS)
print("Clases (senal):", SIGNAL_CLASSES)

## El dataset

Miramos la distribución de entidades y de clases, porque **condiciona qué métrica tiene sentido**
(volvemos sobre esto más abajo).

In [ ]:
from collections import Counter

splits = {}
for s in ["train", "val", "test"]:
    splits[s] = {
        "ner": list(read_jsonl(ROOT / f"data/annotated/ner_{s}.jsonl")),
        "cls": list(read_jsonl(ROOT / f"data/annotated/cls_{s}.jsonl")),
    }
    print(f"{s:6s} NER={len(splits[s]['ner']):5d}   CLS={len(splits[s]['cls']):5d}")

print("\nEntidades en train (conteo de spans):")
ent = Counter(t[2:] for r in splits["train"]["ner"] for t in r["ner_tags"] if t.startswith("B-"))
for k, v in ent.most_common():
    print(f"  {k:14s} {v:6d}")

print("\nSenales en train (un aviso puede tener varias):")
sig = Counter(s for r in splits["train"]["cls"] for s in r.get("signals", []))
n = len(splits["train"]["cls"])
for k, v in sig.most_common():
    print(f"  {k:16s} {v:6d}  ({v/n:.1%} de los avisos)")
sin_senal = sum(1 for r in splits["train"]["cls"] if not r.get("signals"))
print(f"  {'(ninguna)':16s} {sin_senal:6d}  ({sin_senal/n:.1%})")

In [ ]:
# Un ejemplo concreto: como se ve una descripcion ya etiquetada
from src.utils.text import group_entities

ej = splits["train"]["ner"][0]
print("TEXTO:\n ", " ".join(ej["tokens"])[:400], "...\n")
print("ENTIDADES ANOTADAS:")
for e in group_entities(list(zip(ej["tokens"], ej["ner_tags"]))):
    print(f"  {e['type']:14s} -> {e['text']}")

## Fine-tuning

Ambos modelos parten de **BETO** (`dccuchile/bert-base-spanish-wwm-cased`), un BERT entrenado
en español, y se ajustan a cada tarea.

La configuración está condicionada por el hardware disponible — una **GTX 1650 de 4 GB**:
`fp16` para reducir memoria, batch chico, y *gradient accumulation* para recuperar un batch
efectivo razonable sin ocupar más VRAM.

In [ ]:
for tarea, key in [("NER", "ner"), ("Clasificacion", "classifier")]:
    c = cfg[key]
    print(f"--- {tarea} ---")
    print(f"  batch={c['batch_size']} x grad_accum={c['grad_accum']} "
          f"-> batch efectivo {c['batch_size'] * c['grad_accum']}")
    print(f"  epochs={c['epochs']}  lr={c['lr']}  max_length={c['max_length']}  fp16={c['fp16']}")

El entrenamiento se lanza fuera del notebook (tarda varios minutos en GPU):

```bash
python -m src.models.train_ner
python -m src.models.train_classifier
```

---
# Métricas: definición y justificación

### NER — F1 por entidad con `seqeval`, **no** accuracy por token

Dos razones:

1. **El accuracy por token miente.** La enorme mayoría de los tokens son `O` (texto sin entidad),
   así que un modelo que no detecte nada igual saca un accuracy altísimo.
2. **Lo que importa es el span completo.** Si la anotación dice `a estrenar` y el modelo marca sólo
   `estrenar`, para el uso posterior (armar la variable *estado de la propiedad*) eso está mal.
   `seqeval` cuenta una entidad como correcta sólo si coinciden **el tipo y los límites exactos**,
   que es el criterio estricto y el que corresponde acá.

### Clasificación — F1 **macro** además del micro

Las clases están desbalanceadas (ver la distribución de arriba). El F1 micro queda dominado por
las clases frecuentes; el **macro promedia las cuatro clases con igual peso**, así que penaliza
que el modelo ignore una clase rara. Y justamente las señales raras —`URGENCIA`, `OPORTUNIDAD`—
son las más interesantes para detectar subvaluación, que es el propósito del proyecto.

Se reporta también el F1 **por clase**, porque el promedio solo esconde en cuál falla.

In [ ]:
# Metricas de test guardadas por el entrenamiento
for tarea, d in [("NER", cfg["ner"]["out_dir"]), ("CLS", cfg["classifier"]["out_dir"])]:
    p = ROOT / d / "test_metrics.json"
    if p.exists():
        m = json.loads(p.read_text(encoding="utf-8"))
        print(f"--- {tarea} (test) ---")
        for k, v in m.items():
            if isinstance(v, float) and not k.startswith("eval_runtime"):
                print(f"  {k:34s} {v:.4f}")
    else:
        print(f"--- {tarea}: sin entrenar todavia (falta {p}) ---")

In [ ]:
# Reportes detallados por entidad / por clase
for r in sorted(Path("reports").glob("*.md")):
    print("=" * 70)
    print(r.read_text(encoding="utf-8"))

---
# Generalización sintético → real

**Este es el resultado más importante del trabajo, y conviene leerlo con cuidado.**

El F1 sobre el test sintético es altísimo. Eso *no* significa que el modelo sea excelente:
significa que **el dataset sintético es fácil**. Los avisos se generan a partir de plantillas,
así que el modelo puede aprender la plantilla en lugar del concepto.

La prueba honesta es correr el modelo sobre las descripciones **reales** scrapeadas de ZonaProp,
que son mucho más largas y tienen prosa desordenada, abreviaturas y direcciones.

In [ ]:
from src.models.infer import extract_entities
from collections import Counter

reales = list(read_jsonl(ROOT / "data/raw/sample_zonaprop_caba.jsonl"))
sinteticos = list(read_jsonl(ROOT / "data/synthetic/listings.jsonl"))

largo_real = sum(len(r["description"]) for r in reales) // len(reales)
largo_sint = sum(len(r["description"]) for r in sinteticos) // len(sinteticos)
print(f"Avisos reales scrapeados : {len(reales)}")
print(f"Largo medio de descripcion: real={largo_real} vs sintetico={largo_sint} caracteres")

if (ROOT / cfg["ner"]["out_dir"]).exists():
    md, ml = cfg["ner"]["out_dir"], cfg["ner"]["max_length"]
    tipos, por_aviso = Counter(), []
    for r in reales[:10]:
        ents = extract_entities(r["description"][:900], md, ml)
        por_aviso.append(len(ents))
        tipos.update(e["type"] for e in ents)
    print(f"\nEntidades detectadas por aviso real: promedio {sum(por_aviso)/len(por_aviso):.1f}")
    print(f"Distribucion por tipo: {dict(tipos)}")
else:
    print("\nModelos no entrenados todavia.")

In [ ]:
---
# Limitaciones

Se documentan explícitamente, como pedía la propuesta:

1. **Anti-bot: el límite real del scrape.** ZonaProp protege el sitio con Cloudflare.
   Las **páginas de detalle** devuelven un *challenge* en lugar del contenido, así que la
   extracción se hace desde las **tarjetas del listado** — que ya traen la descripción completa
   y además rinden 25-30 avisos por request en vez de uno. Las **URLs paginadas** también quedan
   bloqueadas, incluso subiendo los delays a 20-40 segundos; no se insistió contra el bloqueo.
   Por eso el volumen quedó muy por debajo de los 8.000–15.000 avisos de la propuesta.

2. **Datos sintéticos para entrenar.** Los modelos se entrenaron sobre un generador con etiquetas
   *gold por construcción*, y los avisos reales se usan como evaluación externa. Como se vio más
   arriba, el F1 perfecto sobre sintético **no** se traslada a texto real.

3. **Anotación semiautomática.** El pre-anotador es un LLM local (`llama3.2:3b`), un modelo chico
   elegido por la restricción de 4 GB de VRAM. Sus errores se propagan a las etiquetas; por eso
   la revisión manual del subconjunto.

4. **Longitud del texto.** `max_length` es 192 tokens, así que las descripciones reales
   (mediana ~1.600 caracteres) se truncan. Subirlo no entra en 4 GB de VRAM con este batch.

5. **Términos de uso.** El scraping fue de uso académico, con rate-limiting y sin redistribuir
   contenido del portal: se incluye una muestra acotada de datos ya estructurados, no volcados
   de páginas.

# Trabajo futuro

Las variables generadas por esta capa de NLP están pensadas para integrarse, en el marco de la
**tesis de la maestría**, como *features* de un modelo hedónico de valuación, para detectar
activos subvaluados en CABA. Esa integración excede el alcance de este Trabajo Final.

El paso inmediato sería **anotar un conjunto real grande**: es lo que separa este pipeline
funcionando de un modelo que se pueda usar en producción.

### Qué aprendimos de esto

La primera versión del generador producía descripciones de ~220 caracteres, sin tildes y siempre
con la misma estructura. El modelo alcanzaba F1 = 1.0 sobre ese test, pero al aplicarlo a texto
real etiquetaba como `AMENITY` prácticamente cualquier sustantivo — «universidades», «avenidas»,
«ventilación» — y llegaba a marcar el nombre de una calle como `ORIENTACION`.

El diagnóstico: el modelo no había aprendido *qué es un amenity*, sino *dónde suele aparecer uno*
dentro de la plantilla. Nunca había visto un sustantivo que **no** fuera entidad.

Sobre esa base se rehízo el generador:

| Cambio | Por qué |
|---|---|
| Tildes en todo el vocabulario | Los avisos reales las usan y BETO es un modelo *cased*: distingue `balcón` de `balcon` |
| Oraciones distractoras sin entidades | Para que aprenda a predecir `O`; los tokens `O` pasaron a ser el 90% |
| Orden de secciones barajado | Si el orden es fijo, el modelo aprende la posición en vez del contenido |
| Frases de entrada variadas | No siempre «Cuenta con» antes de los amenities |
| Descripciones más largas | ~586 caracteres, más cerca de los ~1.600 reales |

**La conclusión no se maquilla:** los datos sintéticos sirven para validar que el pipeline
funciona de punta a punta, pero **no sustituyen anotación real**. Un F1 perfecto sobre datos
generados por uno mismo dice más sobre lo fácil que es el test que sobre la calidad del modelo.

---
# El producto final: texto libre → variables estructuradas

Esto es la salida de valor del proyecto. Una descripción cualquiera entra como texto y sale
convertida en atributos que un modelo de valuación podría consumir como *features*.

In [ ]:
from src.models.infer import enrich

EJEMPLOS = [
    "Excelente 3 ambientes a estrenar al frente, con balcon aterrazado, pileta y cochera. "
    "Expensas $95.000. Dueno directo, escucho ofertas.",
    "Departamento a reciclar en Almagro, contrafrente, 40 anios de antiguedad. "
    "Necesita refaccion integral. Venta urgente por mudanza.",
]

if (ROOT / cfg["ner"]["out_dir"]).exists():
    for t in EJEMPLOS:
        out = enrich(t, cfg)
        print("TEXTO:", t[:90], "...")
        print("  entidades:")
        for e in out["entities"]:
            print(f"    {e['type']:14s} -> {e['text']}")
        print(f"  senales  : {out['signals'] or '(ninguna)'}\n")
else:
    print("Modelos no entrenados todavia. Corre:  scripts\\run_demo.bat --quick")

---
# Limitaciones

Se documentan explícitamente, como pedía la propuesta:

1. **Volumen del scrape.** La propuesta apuntaba a 8.000–15.000 avisos; se trabajó con un volumen
   mucho menor. ZonaProp tiene protección anti-bot (DataDome/Cloudflare) y el rate-limiting cortés
   que aplicamos —delays de 3 a 7 segundos— hace que una corrida completa lleve muchas horas.
   El objetivo acá fue **demostrar que la arquitectura funciona**, no maximizar el dataset.

2. **Datos sintéticos para entrenar.** Los modelos se entrenaron sobre un generador de avisos
   sintéticos con etiquetas *gold por construcción*, y el conjunto real anotado se usó como
   evaluación externa. Esto mide generalización sintético → real, pero un dataset real y grande
   daría mejores resultados.

3. **Anotación semiautomática.** El pre-anotador es un LLM local (`llama3.2:3b`), un modelo chico
   elegido por la restricción de 4 GB de VRAM. Sus errores se propagan a las etiquetas; por eso
   la revisión manual del subconjunto.

4. **Términos de uso.** El scraping fue de uso académico, con rate-limiting y sin redistribuir
   contenido del portal.

# Trabajo futuro

Las variables generadas por esta capa de NLP están pensadas para integrarse, en el marco de la
**tesis de la maestría**, como *features* de un modelo hedónico de valuación, para detectar
activos subvaluados en CABA. Esa integración excede el alcance de este Trabajo Final.